In [1]:
#Import libraries
import requests
import pandas as pd
import json
from pathlib import Path

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "Notebooks" else cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Project root: C:\Users\iniob\Toni_projects\Hydrogen Project
Raw folder: C:\Users\iniob\Toni_projects\Hydrogen Project\data\raw
Processed folder: C:\Users\iniob\Toni_projects\Hydrogen Project\data\processed


In [3]:
#Use the same dates as carbon intensity
carbon_path = PROCESSED_DIR / 'carbon_intensity_last_7_complete_days.csv'
df_carbon = pd.read_csv(carbon_path)

df_carbon.head()

,date_requested,from_utc,to_utc,forecast_gco2_per_kwh,actual_gco2_per_kwh,index,from_uk,to_uk,date_uk,hour_uk,time_uk
0,2026-06-22,2026-06-21 23:00:00+00:00,2026-06-21 23:30:00+00:00,193,189,high,2026-06-22 00:00:00+01:00,2026-06-22 00:30:00+01:00,2026-06-22,0,00:00:00
1,2026-06-22,2026-06-21 23:30:00+00:00,2026-06-22 00:00:00+00:00,188,187,high,2026-06-22 00:30:00+01:00,2026-06-22 01:00:00+01:00,2026-06-22,0,00:30:00
2,2026-06-22,2026-06-22 00:00:00+00:00,2026-06-22 00:30:00+00:00,196,182,high,2026-06-22 01:00:00+01:00,2026-06-22 01:30:00+01:00,2026-06-22,1,01:00:00
3,2026-06-22,2026-06-22 00:30:00+00:00,2026-06-22 01:00:00+00:00,190,179,high,2026-06-22 01:30:00+01:00,2026-06-22 02:00:00+01:00,2026-06-22,1,01:30:00
4,2026-06-22,2026-06-22 01:00:00+00:00,2026-06-22 01:30:00+00:00,189,179,high,2026-06-22 02:00:00+01:00,2026-06-22 02:30:00+01:00,2026-06-22,2,02:00:00


In [4]:
dates = sorted(df_carbon["date_requested"].unique())

dates

['2026-06-22',
 '2026-06-23',
 '2026-06-24',
 '2026-06-25',
 '2026-06-26',
 '2026-06-27',
 '2026-06-28']

In [5]:
#Testing an api call
ELEXON_BASE_URL = 'https://data.elexon.co.uk/bmrs/api/v1'

test_date= dates[0]

url = f'{ELEXON_BASE_URL}/balancing/settlement/system-prices/{test_date}'

response = requests.get(url)

response.status_code

200

In [6]:
#Convert to json and inspect
price_data = response.json()

type(price_data)

dict

In [7]:
price_data.keys()

dict_keys(['metadata', 'data'])

In [8]:
price_data['data'][:2]

[{'settlementDate': '2026-06-22',
  'settlementPeriod': 1,
  'startTime': '2026-06-21T23:00:00Z',
  'createdDateTime': '2026-06-22T23:44:40Z',
  'systemSellPrice': 111.19,
  'systemBuyPrice': 111.19,
  'bsadDefaulted': False,
  'priceDerivationCode': 'P',
  'reserveScarcityPrice': 0.0,
  'netImbalanceVolume': 118.45084479166667,
  'sellPriceAdjustment': 0.0,
  'buyPriceAdjustment': 0.0,
  'replacementPrice': 111.19,
  'replacementPriceReferenceVolume': 0.0,
  'totalAcceptedOfferVolume': 238.0,
  'totalAcceptedBidVolume': -119.740821875,
  'totalAdjustmentSellVolume': 0.0,
  'totalAdjustmentBuyVolume': 0.0,
  'totalSystemTaggedAcceptedOfferVolume': 237.0,
  'totalSystemTaggedAcceptedBidVolume': -119.740821875,
  'totalSystemTaggedAdjustmentSellVolume': None,
  'totalSystemTaggedAdjustmentBuyVolume': None},
 {'settlementDate': '2026-06-22',
  'settlementPeriod': 2,
  'startTime': '2026-06-21T23:30:00Z',
  'createdDateTime': '2026-06-23T00:14:45Z',
  'systemSellPrice': 108.63,
  'systemBu

In [9]:
df_prices_day = pd.DataFrame(price_data['data'])

df_prices_day.columns

Index(['settlementDate', 'settlementPeriod', 'startTime', 'createdDateTime',
       'systemSellPrice', 'systemBuyPrice', 'bsadDefaulted',
       'priceDerivationCode', 'reserveScarcityPrice', 'netImbalanceVolume',
       'sellPriceAdjustment', 'buyPriceAdjustment', 'replacementPrice',
       'replacementPriceReferenceVolume', 'totalAcceptedOfferVolume',
       'totalAcceptedBidVolume', 'totalAdjustmentSellVolume',
       'totalAdjustmentBuyVolume', 'totalSystemTaggedAcceptedOfferVolume',
       'totalSystemTaggedAcceptedBidVolume',
       'totalSystemTaggedAdjustmentSellVolume',
       'totalSystemTaggedAdjustmentBuyVolume'],
      dtype='str')

KeyError: 'settlement_date'

In [11]:
df_prices_day = df_prices_day[
    [
        "settlementDate",
        "settlementPeriod",
        "systemBuyPrice",
        "systemSellPrice"
    ]
].copy()

df_prices_day = df_prices_day.rename(columns={
    "settlementDate": "settlement_date",
    "settlementPeriod": "settlement_period",
    "systemBuyPrice": "system_buy_price_gbp_per_mwh",
    "systemSellPrice": "system_sell_price_gbp_per_mwh"
})

df_prices_day.head()

,settlement_date,settlement_period,system_buy_price_gbp_per_mwh,system_sell_price_gbp_per_mwh
0,2026-06-22,1,111.19,111.19
1,2026-06-22,2,108.63,108.63
2,2026-06-22,3,104.94,104.94
3,2026-06-22,4,100.83,100.83
4,2026-06-22,5,81.33,81.33


In [12]:
df_prices_day["settlement_date"] = pd.to_datetime(df_prices_day["settlement_date"])

df_prices_day["from_uk_naive"] = (
    df_prices_day["settlement_date"]
    + pd.to_timedelta((df_prices_day["settlement_period"] - 1) * 30, unit="m")
)

df_prices_day["to_uk_naive"] = df_prices_day["from_uk_naive"] + pd.Timedelta(minutes=30)

df_prices_day.head()

,settlement_date,settlement_period,system_buy_price_gbp_per_mwh,system_sell_price_gbp_per_mwh,from_uk_naive,to_uk_naive
0,2026-06-22,1,111.19,111.19,2026-06-22 00:00:00,2026-06-22 00:30:00
1,2026-06-22,2,108.63,108.63,2026-06-22 00:30:00,2026-06-22 01:00:00
2,2026-06-22,3,104.94,104.94,2026-06-22 01:00:00,2026-06-22 01:30:00
3,2026-06-22,4,100.83,100.83,2026-06-22 01:30:00,2026-06-22 02:00:00
4,2026-06-22,5,81.33,81.33,2026-06-22 02:00:00,2026-06-22 02:30:00


In [13]:
df_prices_day["price_gbp_per_mwh"] = df_prices_day["system_buy_price_gbp_per_mwh"]

df_prices_day.head()

,settlement_date,settlement_period,system_buy_price_gbp_per_mwh,system_sell_price_gbp_per_mwh,from_uk_naive,to_uk_naive,price_gbp_per_mwh
0,2026-06-22,1,111.19,111.19,2026-06-22 00:00:00,2026-06-22 00:30:00,111.19
1,2026-06-22,2,108.63,108.63,2026-06-22 00:30:00,2026-06-22 01:00:00,108.63
2,2026-06-22,3,104.94,104.94,2026-06-22 01:00:00,2026-06-22 01:30:00,104.94
3,2026-06-22,4,100.83,100.83,2026-06-22 01:30:00,2026-06-22 02:00:00,100.83
4,2026-06-22,5,81.33,81.33,2026-06-22 02:00:00,2026-06-22 02:30:00,81.33


In [14]:
def fetch_system_prices_for_date(date_str):
    """
    Fetch and clean Elexon settlement system prices for one date.
    """

    url = f"{ELEXON_BASE_URL}/balancing/settlement/system-prices/{date_str}"
    
    response = requests.get(url)
    response.raise_for_status()
    
    price_data = response.json()
    
    df = pd.DataFrame(price_data["data"])
    
    df = df[
        [
            "settlementDate",
            "settlementPeriod",
            "systemBuyPrice",
            "systemSellPrice"
        ]
    ].copy()
    
    df = df.rename(columns={
        "settlementDate": "settlement_date",
        "settlementPeriod": "settlement_period",
        "systemBuyPrice": "system_buy_price_gbp_per_mwh",
        "systemSellPrice": "system_sell_price_gbp_per_mwh"
    })
    
    df["settlement_date"] = pd.to_datetime(df["settlement_date"])
    
    df["from_uk_naive"] = (
        df["settlement_date"]
        + pd.to_timedelta((df["settlement_period"] - 1) * 30, unit="m")
    )
    
    df["to_uk_naive"] = df["from_uk_naive"] + pd.Timedelta(minutes=30)
    
    df["price_gbp_per_mwh"] = df["system_buy_price_gbp_per_mwh"]
    
    return df

In [15]:
all_price_days = []

for date_str in dates:
    print("Fetching:", date_str)
    df_day = fetch_system_prices_for_date(date_str)
    all_price_days.append(df_day)

df_prices_week = pd.concat(all_price_days, ignore_index=True)


Fetching: 2026-06-22
Fetching: 2026-06-23
Fetching: 2026-06-24
Fetching: 2026-06-25
Fetching: 2026-06-26
Fetching: 2026-06-27
Fetching: 2026-06-28


In [16]:
df_prices_week.shape

(336, 7)

In [17]:
price_path = PROCESSED_DIR / "elexon_system_prices_last_7_complete_days.csv"

df_prices_week.to_csv(price_path, index=False)

print("Saved to:", price_path)

Saved to: C:\Users\iniob\Toni_projects\Hydrogen Project\data\processed\elexon_system_prices_last_7_complete_days.csv
